# Ocean Depth-Time Drift

Depth-time Hovmoller diagrams and centennial drift profiles for horizontally-
averaged ocean temperature and salinity.  Data come from the `ocean_annual_z`
post-processing stream (`thetao_xyave` and `so_xyave`), which are already
horizontally averaged in the PP pipeline.

Authors: John Krasting

In [ ]:
# Development mode: constantly refreshes module code
%load_ext autoreload
%autoreload 2

## Framework Code and Diagnostic Setup

In [ ]:
import esnb
from esnb import NotebookDiagnostic, RequestedVariable, CaseGroup2, nbtools
from esnb.sites.gfdl import call_dmget
import climplot

In [ ]:
# Define requested variables: horizontally-averaged T and S from ocean_annual_z
variables = [
    RequestedVariable("thetao_xyave", "ocean_annual_z", frequency="yearly"),
    RequestedVariable("so_xyave", "ocean_annual_z", frequency="yearly"),
]

# Diagnostic configuration
diag_name = "Ocean Depth-Time Drift"
diag_desc = "Depth-time Hovmoller and centennial drift profiles for temperature and salinity"
user_options = {}

# Initialize the diagnostic
diag = NotebookDiagnostic(diag_name, diag_desc, variables=variables, **user_options)

In [ ]:
# Define experiments to analyze
groups = [
    CaseGroup2("odiv-1", date_range=("0001-01-01", "0500-12-31"), name="CM4.0"),
    CaseGroup2("esm45-122", date_range=("0001-01-01", "0500-12-31"), name="ESM4.5 Spinup"),
    CaseGroup2("esm45-137", date_range=("0001-01-01", "0200-12-31"), name="ESM4.5 piControl"),
    CaseGroup2("esm45-148", date_range=("0001-01-01", "0200-12-31"), name="ESM4.5 piControl v2"),
]

In [ ]:
# Combine experiments with the diagnostic request and determine files to load
diag.resolve(groups)

In [ ]:
# Print a list of file paths
_ = [print(x) for x in diag.files]

*(The files above are necessary to run the diagnostic.)*

In [ ]:
# Stage files from mass storage
call_dmget(diag.files, status=True)
call_dmget(diag.files)

In [ ]:
# Load the data as xarray datasets
diag.open()

## Main Diagnostic

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from momgrid.VerticalSplitScale import VerticalSplitScale
from momgrid.util import infer_bounds

In [ ]:
all_figs = []
climplot.publication()

In [ ]:
# Configuration
SPLITSCALE_ZVAL = [6500, 1500, 0]
SPLIT_DEPTH = 1500
YTICKS = [300, 600, 900, 1200, 1500, 2500, 3500, 4500, 5500, 6500]
CENTURY = 100  # years per profile window

# Compute drift for each group
drift_data = {}
thetao_var = diag.varmap["thetao_xyave"]
so_var = diag.varmap["so_xyave"]

for group in diag.groups:
    ds_t = group.datasets[thetao_var]
    ds_s = group.datasets[so_var]

    # Extract data arrays
    thetao = ds_t["thetao_xyave"]
    so = ds_s["so_xyave"]

    # Compute drift: subtract first time step
    thetao_drift = thetao - thetao.isel(time=0)
    so_drift = so - so.isel(time=0)

    # Time axis as years since start (integer)
    years = np.arange(len(thetao_drift.time))

    # Depth axis
    z_l = thetao.z_l.values

    drift_data[group] = {
        "thetao_drift": thetao_drift,
        "so_drift": so_drift,
        "years": years,
        "z_l": z_l,
    }

In [ ]:
def plot_hovmoller(ax, years, z_l, drift, title, cmap, norm, levels):
    """Depth-time Hovmoller with splitscale and contour overlay."""
    cf = ax.contourf(years, z_l, drift.T, levels=levels,
                     cmap=cmap, norm=norm, extend="both")
    ax.contour(years, z_l, drift.T, levels=levels,
               colors="k", linewidths=0.3)
    ax.set_yscale("splitscale", zval=SPLITSCALE_ZVAL)
    ax.axhline(y=SPLIT_DEPTH, color="k", linestyle="dashed", linewidth=0.5)
    ax.set_yticks(YTICKS)
    ax.set_ylabel("Depth (m)")
    ax.set_xlabel("Year")
    ax.set_title(title)
    return cf

In [ ]:
# Temperature Hovmoller diagrams
cmap_t, norm_t, levels_t = climplot.anomaly_cmap(-0.5, 0.5, 0.025)

for group in diag.groups:
    d = drift_data[group]
    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    cf = plot_hovmoller(ax, d["years"], d["z_l"],
                        d["thetao_drift"].values,
                        f"{group.name} — Temperature Drift (°C)",
                        cmap_t, norm_t, levels_t)
    plt.subplots_adjust(bottom=0.18)
    cax = fig.add_axes([0.15, 0.05, 0.7, 0.03])
    fig.colorbar(cf, cax=cax, orientation="horizontal",
                 extend="both", label="Temperature Drift (°C)")
    all_figs.append(fig)

In [ ]:
# Salinity Hovmoller diagrams
cmap_s, norm_s, levels_s = climplot.anomaly_cmap(-0.25, 0.25, 0.0125)

for group in diag.groups:
    d = drift_data[group]
    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    cf = plot_hovmoller(ax, d["years"], d["z_l"],
                        d["so_drift"].values,
                        f"{group.name} — Salinity Drift (PSU)",
                        cmap_s, norm_s, levels_s)
    plt.subplots_adjust(bottom=0.18)
    cax = fig.add_axes([0.15, 0.05, 0.7, 0.03])
    fig.colorbar(cf, cax=cax, orientation="horizontal",
                 extend="both", label="Salinity Drift (PSU)")
    all_figs.append(fig)

In [ ]:
# Temperature drift profiles (centennial windows)
nexps = len(diag.groups)
fig, axes = plt.subplots(1, nexps, figsize=(3.5 * nexps, 4.0), sharey=True)

# First pass: find global min/max across all experiments and time windows
global_min = np.inf
global_max = -np.inf

for group in diag.groups:
    d = drift_data[group]
    nyears = len(d["years"])
    
    for i in range(0, nyears, CENTURY):
        end = min(i + CENTURY, nyears)
        profile = d["thetao_drift"].isel(time=slice(i, end)).mean("time").values
        global_min = min(global_min, np.nanmin(profile))
        global_max = max(global_max, np.nanmax(profile))

# Add 5% padding to the limits
range_span = global_max - global_min
padding = range_span * 0.05
xlim_min = global_min - padding
xlim_max = global_max + padding

# Second pass: plot with consistent limits
for n, group in enumerate(diag.groups):
    ax = axes[n] if nexps > 1 else axes
    d = drift_data[group]
    nyears = len(d["years"])

    for i in range(0, nyears, CENTURY):
        end = min(i + CENTURY, nyears)
        label = f"yrs {i+1}–{end}"
        profile = d["thetao_drift"].isel(time=slice(i, end)).mean("time")
        ax.plot(profile.values, d["z_l"], label=label)

    ax.set_xlim(xlim_min, xlim_max)  # Apply consistent x-axis limits
    ax.invert_yaxis()
    ax.set_ylim(6500, 0)
    ax.grid(linewidth=0.5, alpha=0.5)
    ax.legend(fontsize=7)
    ax.set_title(group.name)
    if n == 0:
        ax.set_ylabel("Depth (m)")
    ax.set_xlabel("°C")

fig.suptitle("Temperature Drift Profiles")
plt.tight_layout()
all_figs.append(fig)

In [ ]:
# Salinity drift profiles (centennial windows)
nexps = len(diag.groups)
fig, axes = plt.subplots(1, nexps, figsize=(3.5 * nexps, 4.0), sharey=True)

# First pass: find global min/max across all experiments and time windows
global_min = np.inf
global_max = -np.inf

for group in diag.groups:
    d = drift_data[group]
    nyears = len(d["years"])
    
    for i in range(0, nyears, CENTURY):
        end = min(i + CENTURY, nyears)
        profile = d["so_drift"].isel(time=slice(i, end)).mean("time").values
        global_min = min(global_min, np.nanmin(profile))
        global_max = max(global_max, np.nanmax(profile))

# Add 5% padding to the limits
range_span = global_max - global_min
padding = range_span * 0.05
xlim_min = global_min - padding
xlim_max = global_max + padding

# Second pass: plot with consistent limits
for n, group in enumerate(diag.groups):
    ax = axes[n] if nexps > 1 else axes
    d = drift_data[group]
    nyears = len(d["years"])

    for i in range(0, nyears, CENTURY):
        end = min(i + CENTURY, nyears)
        label = f"yrs {i+1}–{end}"
        profile = d["so_drift"].isel(time=slice(i, end)).mean("time")
        ax.plot(profile.values, d["z_l"], label=label)

    ax.set_xlim(xlim_min, xlim_max)  # Apply consistent x-axis limits
    ax.invert_yaxis()
    ax.set_ylim(6500, 0)
    ax.grid(linewidth=0.5, alpha=0.5)
    ax.legend(fontsize=7)
    ax.set_title(group.name)
    if n == 0:
        ax.set_ylabel("Depth (m)")
    ax.set_xlabel("PSU")

fig.suptitle("Salinity Drift Profiles")
plt.tight_layout()
all_figs.append(fig)

In [ ]:
# Extract depth-weighted drift metrics per group
for group in diag.groups:
    d = drift_data[group]
    z_l = d["z_l"]
    z_i = infer_bounds(z_l, start=0.0)
    dz = np.diff(z_i)

    for varname, drift_key in [("thetao_drift", "thetao_drift"),
                                ("so_drift", "so_drift")]:
        drift_arr = d[drift_key]

        # Last-decade mean drift profile
        last_decade = drift_arr.isel(time=slice(-10, None)).mean("time").values

        # Upper ocean (0-1500 m) depth-weighted mean
        upper = z_l <= 1500
        if upper.any():
            upper_drift = np.average(last_decade[upper], weights=dz[upper])
            group.add_metric(varname, ("upper_mean", round(float(upper_drift), 4)))

        # Deep ocean (>1500 m) depth-weighted mean
        deep = z_l > 1500
        if deep.any():
            deep_drift = np.average(last_decade[deep], weights=dz[deep])
            group.add_metric(varname, ("deep_mean", round(float(deep_drift), 4)))

        # Max absolute drift and its depth
        max_idx = np.argmax(np.abs(last_decade))
        group.add_metric(varname, ("max_abs", round(float(last_decade[max_idx]), 4)))
        group.add_metric(varname, ("max_abs_depth_m", round(float(z_l[max_idx]), 1)))

## Write Metrics

In [ ]:
diag.write_metrics("depth_time_drift_metrics.json")

## Save PowerPoint

In [ ]:
nbtools.save_pptx(all_figs, "depth_time_drift.pptx")